# 11 — Risiko pesisir

Untuk kota pesisir seperti Semarang, genangan rob dan penurunan tanah adalah pertanyaan utama. Notebook ini:
DEM dan genangan skenario yang **terhubung ke laut**, skenario penurunan tanah (ilustratif), perubahan garis
pantai dari Landsat, lalu indeks risiko per kelurahan — beserta batas-batasnya.

In [ ]:
import sys
sys.path.insert(0, "../scripts")

import geopandas as gpd
import numpy as np
import pandas as pd

import gee
from kota import CFG, OSM, PROC

ee = gee.init()
kota = gpd.read_file(OSM, layer="batas_kota")
AOI = gee.to_ee(kota, simplify_m=30).geometry()
ZONA = AOI.buffer(3000).bounds()  # kota + laut di depannya
kel = gpd.read_file(PROC / "morfologi.gpkg", layer="kelurahan")
fc = gee.to_ee(kel.to_crs(4326), cols=["id", "kelurahan", "kecamatan"])

## 1. DEM
Copernicus GLO-30: 30 m, tinggi terhadap geoid EGM2008 (≈ muka laut rata-rata). Ini **model permukaan**
(DSM): atap bangunan dan tajuk pohon ikut terukur, sehingga kawasan padat tampak lebih tinggi dari tanahnya.

In [ ]:
glo = ee.ImageCollection("COPERNICUS/DEM/GLO30_2024_1").filterBounds(ZONA).mosaic() \
    .setDefaultProjection("EPSG:4326", None, 30).clip(ZONA)
dem = glo.select("DEM")
st = dem.reduceRegion(ee.Reducer.percentile([5, 50, 95]), AOI, 30, maxPixels=1e10).getInfo()
print("elevasi kota (m) persentil 5/50/95:", {k: round(v, 1) for k, v in st.items()})
rendah = dem.lt(2).selfMask().multiply(ee.Image.pixelArea()).divide(1e4)
print("luas kota di bawah 2 m:", round(rendah.reduceRegion(ee.Reducer.sum(), AOI, 30, maxPixels=1e10)
                                        .getInfo()["DEM"]), "ha")

## 2. Genangan skenario yang terhubung ke laut
Model *bathtub* sederhana menandai semua piksel di bawah muka air. Itu melebih-lebihkan: cekungan di
pedalaman yang tidak tersambung ke laut ikut "tergenang". Kita hanya menghitung piksel yang **terhubung**
ke laut melalui piksel lain yang juga di bawah muka air.

Laut diambil dari pita **WBM** (*water body mask*) DEM yang sama: 1 = laut. (ESA WorldCover tidak bisa dipakai
untuk ini: produk itu hanya mencakup daratan, laut lepas kosong.)

In [ ]:
laut = glo.select("WBM").eq(1).unmask(0)
laut = laut.reproject(crs=f"EPSG:{CFG['epsg_utm']}", scale=30)  # grid tetap 30 m untuk jarak
print("luas laut dalam zona analisis:", round(laut.selfMask().multiply(ee.Image.pixelArea()).divide(1e4)
                                              .reduceRegion(ee.Reducer.sum(), ZONA, 30, maxPixels=1e10)
                                              .getInfo()["WBM"]), "ha")


def genangan(muka_air, dem_img=dem):
    bawah = dem_img.lte(muka_air)
    biaya = ee.Image.constant(1).updateMask(bawah)
    jarak = biaya.cumulativeCost(source=laut.selfMask(), maxDistance=20000)
    return jarak.gte(0).unmask(0).And(bawah).And(laut.Not()).rename("genang")


def luas_ha(img, geom=AOI):
    return img.selfMask().multiply(ee.Image.pixelArea()).divide(1e4) \
        .reduceRegion(ee.Reducer.sum(), geom, 30, maxPixels=1e10).getInfo()["genang"]


baris = {}
for h in [0.5, 1.0, 1.5]:
    bathtub = dem.lte(h).And(laut.Not()).rename("genang")
    baris[f"{h} m"] = {"bathtub (ha)": luas_ha(bathtub), "terhubung ke laut (ha)": luas_ha(genangan(h))}
pd.DataFrame(baris).T.round(0)

## 3. Skenario penurunan tanah (ilustratif)
Pesisir utara Semarang turun beberapa sentimeter per tahun. Laju sebenarnya diukur dengan InSAR atau GNSS;
di sini kita memakai **laju ilustratif** — bukan hasil pengukuran: 8 cm/tahun pada lahan di bawah 2 m,
menurun linier ke 0 pada elevasi 10 m. Proyeksi 10 tahun, muka air tetap 1,0 m.

In [ ]:
laju = ee.Image(0.08).multiply(ee.Image(1).subtract(dem.subtract(2).divide(8).clamp(0, 1)))
dem_2035 = dem.subtract(laju.multiply(10))
g_now, g_2035 = genangan(1.0), genangan(1.0, dem_2035)
pd.Series({"muka air 1,0 m — hari ini (ha)": luas_ha(g_now),
           "muka air 1,0 m — setelah 10 tahun turun (ha)": luas_ha(g_2035)}).round(0)

## 4. Perubahan garis pantai
Air/darat dari MNDWI komposit Landsat 5 (1995–1997) dan Landsat 8/9 (2022–2024). Perubahan dihitung di jalur
pesisir (3 km dari garis pantai saat ini).

In [ ]:
def mndwi_l5(img):
    qa = img.select("QA_PIXEL").bitwiseAnd(0b11111).eq(0)
    return img.select(["SR_B2", "SR_B5"]).multiply(0.0000275).add(-0.2).normalizedDifference().updateMask(qa)


def mndwi_l8(img):
    qa = img.select("QA_PIXEL").bitwiseAnd(0b11111).eq(0)
    return img.select(["SR_B3", "SR_B6"]).multiply(0.0000275).add(-0.2).normalizedDifference().updateMask(qa)


air_lama = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2").filterBounds(ZONA).filterDate("1995-01-01", "1997-12-31") \
    .map(mndwi_l5).median().gt(0)
air_baru = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")) \
    .filterBounds(ZONA).filterDate("2022-01-01", "2024-12-31").map(mndwi_l8).median().gt(0)
jalur = laut.fastDistanceTransform(256).sqrt().multiply(30).lte(3000).clip(ZONA)
jadi_air = air_lama.Not().And(air_baru).And(jalur).rename("genang")
jadi_darat = air_lama.And(air_baru.Not()).And(jalur).rename("genang")
pd.Series({"darat → air, 1995→2024 (ha)": luas_ha(jadi_air, ZONA),
           "air → darat, 1995→2024 (ha)": luas_ha(jadi_darat, ZONA)}).round(0)

"Air → darat" mencakup reklamasi dan pelabuhan; "darat → air" mencakup abrasi, genangan permanen, dan tambak
baru. MNDWI tidak membedakan laut, tambak, dan genangan — periksa visual sebelum menyimpulkan.

## 5. Indeks risiko per kelurahan
Tiga komponen, dinormalisasi 0–1: **bahaya** (bagian luas tergenang pada skenario 1,0 m + 10 tahun turun),
**keterpaparan** (kepadatan bangunan), dan **kedekatan ke garis pantai**. Bobot 0,5 / 0,3 / 0,2.

In [ ]:
area = ee.Image.pixelArea()
stat = ee.Image.cat([g_2035.multiply(area).rename("ha_genang"), area.rename("ha_total"),
                     laut.fastDistanceTransform(512).sqrt().multiply(30).rename("jarak_laut")])
hasil = stat.reduceRegions(fc, ee.Reducer.sum().combine(ee.Reducer.mean(), sharedInputs=True), 30).getInfo()
df = pd.DataFrame([f["properties"] for f in hasil["features"]])
df["bagian_genang"] = df["ha_genang_sum"] / df["ha_total_sum"]
df = kel[["id", "bangunan_per_ha", "geometry"]].merge(df[["id", "kelurahan", "kecamatan", "bagian_genang",
                                                           "jarak_laut_mean"]], on="id")


def norm(s):
    return (s - s.min()) / (s.max() - s.min())


df["indeks_risiko"] = (0.5 * norm(df["bagian_genang"]) + 0.3 * norm(df["bangunan_per_ha"])
                       + 0.2 * (1 - norm(df["jarak_laut_mean"])))
df.sort_values("indeks_risiko", ascending=False)[
    ["kelurahan", "kecamatan", "bagian_genang", "bangunan_per_ha", "jarak_laut_mean", "indeks_risiko"]].head(10).round(3)

**Batas model ini:** DSM, bukan DTM; tanpa tanggul, pompa, dan saluran; laju penurunan ilustratif; bobot
indeks adalah pilihan, bukan fakta. Hasilnya menunjukkan *di mana perlu dilihat lebih dekat*, bukan peta
genangan resmi.

In [ ]:
gpd.GeoDataFrame(df, crs=kel.crs).to_file(PROC / "risiko_pesisir.gpkg", layer="kelurahan")
print("disimpan: risiko_pesisir.gpkg")